# Шаг 2. Записываем голос (Qwen3-TTS)

Этот блокнот делает две вещи:

1. **Придумывает голос по описанию** — вы слушаете 4 варианта и выбираете лучший.
2. **Читает выбранным голосом все фразы** из `phrases.txt` — одним и тем же голосом от первой до последней.

**Перед началом:** на Google Диске должна лежать папка `andrey_voice`, а в ней файл `phrases.txt`
(его делает `VOICE_1_PHRASES.bat` на вашем ПК).

**Как пользоваться:** нажимайте ▶ слева от ячеек **по порядку, сверху вниз**. Каждая ячейка начинается с заголовка.
Если ячейка пишет красную ошибку — прочтите последние строки, там обычно сказано, что делать.

**Сначала выберите видеокарту:** меню «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → «Сохранить».

## 1. Подключаем Google Диск

In [ ]:
#@title Подключить Диск и проверить видеокарту { display-mode: "form" }
import glob
import os
import subprocess
import sys

from IPython.display import Audio, display


def run(*command, cwd=None):
    """Запустить команду и показывать её вывод по мере появления. Ошибка команды останавливает блокнот."""
    process = subprocess.Popen([str(part) for part in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, cwd=cwd)
    for line in process.stdout:
        print(line, end='')
    if process.wait():
        raise RuntimeError('Команда завершилась с ошибкой (строки выше подскажут, с какой). Если не поймёте — '
                           'скопируйте последние 20 строк и покажите их в поддержке.')

from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/andrey_voice'  #@param {type:"string"}
os.makedirs(WORK, exist_ok=True)
os.makedirs('/content/voice_tools', exist_ok=True)

gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('Видеокарта:', gpu or 'НЕ НАЙДЕНА — меню «Среда выполнения» → «Сменить среду выполнения» → T4 GPU → «Сохранить».')

In [ ]:
#@title Проверить файл с фразами { display-mode: "form" }
phrases = os.path.join(WORK, 'phrases.txt')
if os.path.exists(phrases):
    lines = [x for x in open(phrases, encoding='utf-8').read().splitlines() if x.strip() and not x.startswith('#')]
    print(f'phrases.txt найден: {len(lines)} фраз.')
else:
    print('phrases.txt НЕ НАЙДЕН. Загрузите его в папку andrey_voice на Google Диске (drive.google.com) и запустите ячейку снова.')

## 2. Устанавливаем Qwen3-TTS

Занимает 2–5 минут. Сообщения о «несовместимых версиях» можно игнорировать, если ячейка закончилась без слова ERROR в конце.

In [ ]:
#@title Установить Qwen3-TTS { display-mode: "form" }
run(sys.executable, '-m', 'pip', 'install', '-q', '-U', 'qwen-tts', 'soundfile')
print('Готово.')

## 3. Загружаем программы записи

Эта ячейка сама ничего не делает — она только кладёт файл программы рядом с блокнотом.

In [ ]:
%%writefile /content/voice_tools/qwen_generate.py
"""Шаг 2. Записать голос нейросетью Qwen3-TTS (Apache 2.0). Запускается в Google Colab или на ПК с видеокартой NVIDIA.

Две команды:

  design — придумать голос по описанию словами: получится несколько вариантов эталонной фразы
           (reference_1.wav … reference_4.wav). Послушайте и выберите лучший.
  clone  — прочитать ВСЕ фразы выбранным голосом (он остаётся одним и тем же от первой до последней фразы).
           Можно остановить и запустить снова — уже записанное не повторяется.

Примеры:
  python qwen_generate.py design --out /content/drive/MyDrive/andrey_voice
  python qwen_generate.py clone  --out /content/drive/MyDrive/andrey_voice --reference 2 --phrases phrases.txt
"""
import argparse
import hashlib
import sys
import time
from pathlib import Path

LANGUAGE = 'Russian'
DESIGN_MODEL = 'Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign'
BASE_MODELS = {'1.7B': 'Qwen/Qwen3-TTS-12Hz-1.7B-Base', '0.6B': 'Qwen/Qwen3-TTS-12Hz-0.6B-Base'}   # 0.6B — запасной, легче
# Описание голоса пишется по-английски (так его лучше понимает модель). Имён актёров не указывайте.
DEFAULT_INSTRUCT = ('A calm adult male voice, low-pitched and warm, speaking at a measured, unhurried pace with clear '
                    'articulation and a polite, confident tone, like a refined digital assistant.')
DEFAULT_REF_TEXT = 'Добрый вечер. Все системы работают в штатном режиме, и я готов выполнить любую вашу команду.'


def phrase_id(text):
    """Имя файла по тексту: не зависит от порядка строк, поэтому редактирование phrases.txt ничего не ломает."""
    return 'p_' + hashlib.sha1(text.encode('utf-8')).hexdigest()[:10]


def read_phrases(path):
    seen, out = set(), []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if line and not line.startswith('#') and line not in seen:
            seen.add(line)
            out.append(line)
    return out


def save_wav(path, wav, sample_rate):
    import soundfile
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    soundfile.write(str(path), wav, int(sample_rate), subtype='PCM_16')


# ---------------- логика (не зависит от нейросети — проверяется тестами) ----------------
def design_candidates(generate, out_dir, instruct, ref_text, count=4, seed=1234, seed_fn=None, write=save_wav):
    """Несколько вариантов эталонного голоса. generate(text, language, instruct) → (wavs, sample_rate)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    made = []
    for index in range(1, count + 1):
        if seed_fn:
            seed_fn(seed + index)
        wavs, rate = generate(text=ref_text, language=LANGUAGE, instruct=instruct)
        path = out_dir / f'reference_{index}.wav'
        write(path, wavs[0], rate)
        made.append(path)
    (out_dir / 'reference_text.txt').write_text(ref_text, encoding='utf-8')
    (out_dir / 'reference_instruct.txt').write_text(instruct, encoding='utf-8')
    return made


def clone_all(generate_clone, phrases, out_dir, batch=4, write=save_wav, log=print, clock=time.monotonic):
    """Прочитать все фразы. generate_clone(texts) → (wavs, sample_rate). Продолжает с того места, где остановились.

    Если на пачке не хватило памяти видеокарты — пачка делится пополам; фраза, которая не получилась даже одна,
    пропускается и попадает в failed.txt."""
    out_dir = Path(out_dir)
    wav_dir = out_dir / 'wavs'
    wav_dir.mkdir(parents=True, exist_ok=True)
    index_path = out_dir / 'index.tsv'
    failed_path = out_dir / 'failed.txt'
    todo = [p for p in phrases if not (wav_dir / f'{phrase_id(p)}.wav').exists()]
    done_before = len(phrases) - len(todo)
    log(f'Всего фраз: {len(phrases)}, уже записано: {done_before}, осталось: {len(todo)}')
    started, written = clock(), 0

    def run(texts):
        nonlocal written
        try:
            wavs, rate = generate_clone(texts)
        except Exception as error:                      # обычно «CUDA out of memory»
            if len(texts) == 1:
                with failed_path.open('a', encoding='utf-8') as file:
                    file.write(f'{texts[0]}\t{type(error).__name__}: {str(error)[:200]}\n')
                log(f'  пропускаю фразу: {texts[0][:50]}… ({type(error).__name__})')
                return
            half = len(texts) // 2
            run(texts[:half])
            run(texts[half:])
            return
        with index_path.open('a', encoding='utf-8') as index:
            for text, wav in zip(texts, wavs):
                name = f'{phrase_id(text)}.wav'
                write(wav_dir / name, wav, rate)
                index.write(f'{name}\t{text}\n')
                written += 1

    for start in range(0, len(todo), batch):
        run(todo[start:start + batch])
        spent = clock() - started
        if written:
            left = (len(todo) - written) * spent / written
            log(f'  {done_before + written}/{len(phrases)} · осталось примерно {left / 60:.0f} мин')
    return written


# ---------------- нейросеть ----------------
DTYPES = ['auto', 'float16', 'bfloat16', 'float32']


def pick_dtype(choice, modern):
    """'auto' — bfloat16 там, где видеокарта его умеет (Ampere и новее), иначе float16 (Colab T4)."""
    if choice in ('float16', 'bfloat16', 'float32'):
        return choice
    return 'bfloat16' if modern else 'float16'


def load_model(name, dtype_choice='auto'):
    import torch
    from qwen_tts import Qwen3TTSModel
    modern = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    dtype = getattr(torch, pick_dtype(dtype_choice, modern))
    attn = 'sdpa'
    if modern and dtype != torch.float32:
        try:
            import flash_attn  # noqa: F401
            attn = 'flash_attention_2'
        except ImportError:
            pass
    print(f'Загружаю {name} ({str(dtype).split(".")[-1]}, внимание: {attn})…')
    return Qwen3TTSModel.from_pretrained(name, device_map='cuda:0', dtype=dtype, attn_implementation=attn)


def seed_everything(seed):
    import random
    import numpy
    import torch
    random.seed(seed)
    numpy.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def command_design(args):
    model = load_model(DESIGN_MODEL, args.dtype)
    paths = design_candidates(model.generate_voice_design, args.out, args.instruct, args.ref_text, args.count,
                              args.seed, seed_everything)
    print('\nГотово. Послушайте файлы (в Colab: значок папки слева → папка с вашим голосом):')
    for path in paths:
        print('  ', path.name)
    print('\nВыберите лучший и запустите команду clone с его номером, например:  --reference 2')
    print('Не понравился ни один — запустите design ещё раз с другим --seed или другим --instruct.')


def command_clone(args):
    out = Path(args.out)
    reference = out / f'reference_{args.reference}.wav'
    if not reference.exists():
        sys.exit(f'Нет файла {reference}. Сначала запустите команду design и выберите номер голоса.')
    ref_text = (out / 'reference_text.txt').read_text(encoding='utf-8').strip()
    phrases = read_phrases(args.phrases)
    if args.limit:
        phrases = phrases[:args.limit]
    model = load_model(BASE_MODELS[args.model_size], args.dtype)
    prompt = model.create_voice_clone_prompt(ref_audio=str(reference), ref_text=ref_text)

    def generate(texts):
        return model.generate_voice_clone(text=texts, language=[LANGUAGE] * len(texts), voice_clone_prompt=prompt)

    seed_everything(args.seed)
    written = clone_all(generate, phrases, out, args.batch)
    print(f'\nГотово: записано {written} фраз. Дальше — шаг 3 (make_dataset.py).')


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = parser.add_subparsers(dest='command', required=True)
    design = sub.add_parser('design', help='придумать голос по описанию')
    design.add_argument('--out', required=True, help='папка для результатов (в Colab — на Google Диске)')
    design.add_argument('--instruct', default=DEFAULT_INSTRUCT, help='описание голоса по-английски')
    design.add_argument('--ref-text', default=DEFAULT_REF_TEXT, help='что голос произнесёт для пробы (по-русски)')
    design.add_argument('--count', type=int, default=4, help='сколько вариантов сделать')
    design.add_argument('--seed', type=int, default=1234)
    design.add_argument('--dtype', choices=DTYPES, default='auto', help='точность чисел; auto подходит почти всегда')
    clone = sub.add_parser('clone', help='прочитать все фразы выбранным голосом')
    clone.add_argument('--out', required=True)
    clone.add_argument('--reference', type=int, required=True, help='номер понравившегося варианта')
    clone.add_argument('--phrases', required=True, help='файл phrases.txt')
    clone.add_argument('--batch', type=int, default=4, help='фраз за раз (мало памяти — поставьте 2 или 1)')
    clone.add_argument('--model-size', choices=['1.7B', '0.6B'], default='1.7B',
                       help='0.6B — если на 1.7B не хватает памяти видеокарты (качество чуть ниже)')
    clone.add_argument('--limit', type=int, default=0, help='только первые N фраз (для пробы)')
    clone.add_argument('--seed', type=int, default=7)
    clone.add_argument('--dtype', choices=DTYPES, default='auto', help='точность чисел; auto подходит почти всегда')
    args = parser.parse_args(argv)
    {'design': command_design, 'clone': command_clone}[args.command](args)


if __name__ == '__main__':
    main()

## 4. Придумываем голос

Опишите голос **по-английски** (так модель понимает лучше). Не указывайте имена настоящих актёров или персонажей —
голос реального человека копировать нельзя, а описание «как у X» этого и добивается.

Нажмите ▶. Модель скачается (несколько минут, один раз), потом сделает 4 варианта.

In [ ]:
#@title Придумать голос (4 варианта) { display-mode: "form" }
INSTRUCT = "A calm adult male voice, low-pitched and warm, speaking at a measured, unhurried pace with clear articulation and a polite, confident tone, like a refined digital assistant."  #@param {type:"string"}
REF_TEXT = "Добрый вечер. Все системы работают в штатном режиме, и я готов выполнить любую вашу команду."  #@param {type:"string"}
SEED = 1234  #@param {type:"integer"}

run(sys.executable, '/content/voice_tools/qwen_generate.py', 'design', '--out', WORK,
    '--instruct', INSTRUCT, '--ref-text', REF_TEXT, '--seed', SEED)

In [ ]:
#@title Послушать варианты { display-mode: "form" }
for number in range(1, 5):
    path = f'{WORK}/reference_{number}.wav'
    if os.path.exists(path):
        print('Вариант', number)
        display(Audio(path))

Ни один не понравился? Измените описание или число `SEED` в ячейке выше и запустите её снова (старые варианты перезапишутся).
Хотите голос ниже, моложе, быстрее — так и напишите в описании: *low-pitched*, *young*, *faster pace*, *slightly husky*.

## 5. Пробная запись (8 фраз)

Выберите номер понравившегося варианта. Эта ячейка прочитает только 8 фраз, чтобы вы проверили голос,
прежде чем тратить час на все. Прочитанное **не пропадёт** — основная запись продолжится с этого места.

In [ ]:
#@title Пробная запись { display-mode: "form" }
REFERENCE = 1  #@param {type:"integer"}
MODEL_SIZE = "1.7B"  #@param ["1.7B", "0.6B"]

run(sys.executable, '/content/voice_tools/qwen_generate.py', 'clone', '--out', WORK, '--reference', REFERENCE,
    '--phrases', f'{WORK}/phrases.txt', '--limit', 8, '--batch', 4, '--model-size', MODEL_SIZE)
for line in open(f'{WORK}/index.tsv', encoding='utf-8').read().splitlines()[:8]:
    name, _, text = line.partition('\t')
    print(text)
    display(Audio(f'{WORK}/wavs/{name}'))

## 6. Записываем все фразы

Это самая долгая часть. Ориентир — от одного до нескольких часов для ~1000 фраз; точнее скажет сама ячейка
(она пишет «осталось примерно N мин»). **Не закрывайте вкладку.**

* Если Colab отключился или вы остановили ячейку (кнопка ■) — просто запустите её снова:
  уже записанные фразы не повторяются.
* Ошибка «CUDA out of memory» — поставьте `BATCH = 2` или `MODEL_SIZE = "0.6B"`.

In [ ]:
#@title Записать все фразы { display-mode: "form" }
REFERENCE = 1  #@param {type:"integer"}
MODEL_SIZE = "1.7B"  #@param ["1.7B", "0.6B"]
BATCH = 4  #@param {type:"integer"}

run(sys.executable, '/content/voice_tools/qwen_generate.py', 'clone', '--out', WORK, '--reference', REFERENCE,
    '--phrases', f'{WORK}/phrases.txt', '--batch', BATCH, '--model-size', MODEL_SIZE)

In [ ]:
#@title Итог записи { display-mode: "form" }
import soundfile
files = sorted(glob.glob(f'{WORK}/wavs/*.wav'))
seconds = sum(soundfile.info(f).duration for f in files)
print(f'Записано фраз: {len(files)}, всего {seconds / 60:.0f} мин.')
failed = f'{WORK}/failed.txt'
if os.path.exists(failed):
    print('Не получились (можно просто запустить запись ещё раз):', len(open(failed, encoding="utf-8").read().splitlines()))
print('Дальше: откройте блокнот andrey_voice_2_obuchenie.ipynb.')